In [ ]:
from moabb.paradigms import MotorImagery
from moabb.datasets.utils import find_intersecting_channels
from brainbot_dataset import get_brainbot_dataset
from datasets import PhysionetMI16, Weibo2014_16
from utils import print_results_summary

import moabb
import mne

moabb.set_log_level('INFO')
mne.set_log_level('INFO')

SUBJECTS = 6
MAX_TRIALS = 4

brainbot_dataset = get_brainbot_dataset()
brainbot_dataset.n_sessions = min(MAX_TRIALS, brainbot_dataset.n_sessions)
brainbot_dataset.subject_list = brainbot_dataset.subject_list[:SUBJECTS]
physionet16_dataset = PhysionetMI16()
physionet16_dataset.subject_list = physionet16_dataset.subject_list[:SUBJECTS]
weibo2014_16_dataset = Weibo2014_16()
weibo2014_16_dataset.subject_list = weibo2014_16_dataset.subject_list[:SUBJECTS]
assert len(weibo2014_16_dataset.subject_list) == SUBJECTS
assert len(physionet16_dataset.subject_list) == SUBJECTS
assert len(brainbot_dataset.subject_list) == min(len(brainbot_dataset.subject_list), SUBJECTS)
assert brainbot_dataset.n_sessions == MAX_TRIALS


datasets = [brainbot_dataset, physionet16_dataset, weibo2014_16_dataset]
dataset_results = {}
dataset_events = ["left_hand", "right_hand", "feet", "hands", "rest"]
sampling = 160 # based on Physionet sampling rate 

electrodes, datasets = find_intersecting_channels(datasets)
print("Datasets used:", [type(d).__name__ for d in datasets])
print("Used electrodes:", electrodes)

paradigm = MotorImagery(n_classes=len(dataset_events), events=dataset_events, resample=sampling)

### Test all builtin pipelines

In [ ]:
from moabb import benchmark
import os

def run_moabb_benchmark(pipelines_dir, base_dir="./benchmarks", datasets_list=datasets, n_jobs=-1):
    pipelines_path = os.path.join(os.getcwd(), pipelines_dir)
    print(pipelines_path)

    cache_config = dict(
        use=True,
        save_raw=True,
        save_epochs=True,
        save_array=True,
        overwrite_raw=False,
        overwrite_epochs=False,
        overwrite_array=False,
    )

    print("Using cache dir:", mne.get_config('MNE_DATA'))

    return benchmark(
        pipelines=pipelines_path,
        evaluations=["WithinSession"],
        paradigms=["MotorImagery"],
        include_datasets=datasets_list,
        results=os.path.join(base_dir, f"results-{pipelines_dir}"),
        overwrite=False,
        plot=False,
        output=os.path.join(base_dir, f"output-{pipelines_dir}"),
        n_jobs=n_jobs,
        cache_config=cache_config
    )

results = run_moabb_benchmark("pipelines_MI")

In [ ]:
print_results_summary(results)

### Tensorflow pipelines have to be run separately due to OOM issues when running them in parallel

In [ ]:
results = run_moabb_benchmark("pipelines_MI_tensorflow", n_jobs=1)
print_results_summary(results)